# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaant7/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [8]:
import os
from dotenv import load_dotenv
import duckdb

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN is not None, "HF_TOKEN .env dosyasında bulunamadı"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_summary = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS total_sessions,
        SUM(scroll_events) AS total_scroll_events
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

print("Shape:", feature_summary.shape)
feature_summary[["total_impressions", "total_clicks", "avg_position", "total_sessions", "total_scroll_events"]].describe()

Shape: (176738, 6)


,total_impressions,total_clicks,avg_position,total_sessions,total_scroll_events
count,176738.000000,176738.000000,176738.000000,128012.000000,128012.000000
mean,1587.986675,4.650002,15.999277,9.676866,1.582969
std,5431.337724,26.722649,17.686260,39.868167,7.664162
min,1.000000,0.000000,0.000000,0.000000,0.000000
25%,20.000000,0.000000,5.001970,0.000000,0.000000
50%,173.000000,0.000000,8.505296,0.000000,0.000000
75%,1039.000000,2.000000,20.369190,4.000000,1.000000
max,617124.000000,5668.000000,309.000000,2603.000000,667.000000


Distributions show heavy right tails across every signal. total_impressions: median 173, but max 617,124 (594x the 75th percentile) — a small number of pages dominate visibility. total_clicks is even more skewed: median 0, 75th percentile only 2, but max 5,668. The mean (1,588 for impressions) sits far above the median (173), confirming the distribution isn't close to normal — a handful of high-traffic pages pull the average up.

Practical implication: any rule or model using raw sums (like my Week 4 baseline score) will be dominated by these high-volume outliers unless I explicitly account for it — log transforms, percentile-based thresholds, or minimum-volume filters (as the lane guide recommends) are worth considering before trusting raw magnitudes.

Data quality note: avg_position has a minimum of 0.0, which isn't a valid Google ranking position (rankings start at 1) — likely a data artifact worth filtering out (avg_position > 0) before analysis, which I already do in my baseline query.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW daily_agg_march AS
    SELECT content_hash_id,
           SUM(gsc_impressions) AS total_impressions,
           SUM(gsc_clicks) AS total_clicks,
           AVG(gsc_avg_position) AS avg_position,
           SUM(ga4_sessions) AS total_sessions,
           SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""")

staleness_check = con.sql(f"""
    WITH joined AS (
        SELECT d.*, c.content_created_date,
               DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
        FROM daily_agg_march d
        JOIN {TABLES['dim_content']} c ON d.content_hash_id = c.content_hash_id
        WHERE d.total_impressions > 0
    )
    SELECT
        CASE WHEN content_age_days >= 180 THEN 'stale (180+ days)' ELSE 'fresh (<180 days)' END AS age_bucket,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(CASE WHEN imp_second_half < imp_first_half THEN 1 ELSE 0 END) / COUNT(*), 1) AS decline_pct
    FROM joined
    GROUP BY age_bucket
""").df()
staleness_check

,age_bucket,n,decline_pct
0,stale (180+ days),93131,38.6
1,fresh (<180 days),83607,36.6


Signal 1 — Staleness (content_age_days ≥ 180): n=93,131 (stale) vs n=83,607 (fresh). Decline rate 38.6% vs 36.6% — a 2-point gap. Direction matches FlyRank's stale_visible_page flag assumption, but the effect is weak. Verdict: MIXED.

In [11]:
ctr_position_check = con.sql(f"""
    WITH filtered AS (
        SELECT *,
               100.0 * total_clicks / total_impressions AS ctr,
               CASE WHEN avg_position <= 20 THEN 'top20' ELSE 'below20' END AS position_bucket
        FROM daily_agg_march
        WHERE total_impressions >= 500 AND avg_position > 0
    )
    SELECT
        position_bucket,
        COUNT(*) AS n,
        ROUND(AVG(ctr), 2) AS avg_ctr,
        ROUND(100.0 * SUM(CASE WHEN ctr < 0.5 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_low_ctr
    FROM filtered
    GROUP BY position_bucket
""").df()
ctr_position_check

,position_bucket,n,avg_ctr,pct_low_ctr
0,below20,11160,0.13,95.0
1,top20,50764,0.32,81.1


Signal 2 — CTR vs. position (n=50,764 top20 / n=11,160 below20, ≥500 impressions): Avg CTR 0.32% (top20) vs 0.13% (below20). Direction confirmed — CTR drops with worse position, matching the assumption behind FlyRank's low_ctr_visible_page flag. Verdict: CONFIRMED (though the 0.5% threshold catches 81% of even top20 pages — worth a tighter threshold later).

In [12]:
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW quick_win_candidates AS
    SELECT d.content_hash_id, d.avg_position, d.imp_first_half, d.imp_second_half,
           c.search_volume
    FROM daily_agg_march d
    JOIN {TABLES['dim_content']} c ON d.content_hash_id = c.content_hash_id
    WHERE d.avg_position BETWEEN 11 AND 20
      AND d.total_impressions > 0
""")

volume_check = con.sql("""
    SELECT
        CASE WHEN search_volume >= 100 THEN 'high_volume (100+)' ELSE 'low_volume (<100)' END AS volume_bucket,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(CASE WHEN imp_second_half > imp_first_half THEN 1 ELSE 0 END) / COUNT(*), 1) AS rising_pct
    FROM quick_win_candidates
    WHERE search_volume IS NOT NULL
    GROUP BY volume_bucket
""").df()
volume_check

,volume_bucket,n,rising_pct
0,high_volume (100+),2879,57.0
1,low_volume (<100),23357,59.9


Signal 3 — Volume (quick-win assumption, position 11-20, n=23,357 low-volume / n=2,879 high-volume): 59.9% of low-volume pages rose in the second half of March vs. 57.0% of high-volume pages — the opposite direction from what is_quick_win assumes. The gap is small (2.9 points), so I wouldn't call this a strong disproof, but it doesn't support the idea that high search-volume position-11-20 pages are especially likely to "break through." Verdict: OPPOSITE (weak).

Possible explanation: high-volume keywords are also more competitive — more sites are fighting for that position, so movement into the top 10 may be harder, not easier, despite the higher potential reward. Low-volume/low-competition keywords might move more easily simply because there's less competition to break through.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
flag_test = con.sql("""
    WITH scored AS (
        SELECT *,
               100.0 * total_clicks / total_impressions AS ctr,
               CASE
                   WHEN total_impressions >= 500 AND avg_position > 0 AND avg_position <= 20
                        AND (100.0 * total_clicks / total_impressions) < 0.5
                   THEN 'flagged'
                   ELSE 'not_flagged'
               END AS flag_status
        FROM daily_agg_march
        WHERE total_impressions > 0
    )
    SELECT
        flag_status,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(CASE WHEN imp_second_half < imp_first_half THEN 1 ELSE 0 END) / COUNT(*), 1) AS decline_pct
    FROM scored
    GROUP BY flag_status
""").df()
flag_test

,flag_status,n,decline_pct
0,flagged,41169,38.4
1,not_flagged,135569,37.5


Flag-linked test — low_ctr_visible_page: Applying the full flag definition (impressions ≥ 500, position ≤ 20, CTR < 0.5%), flagged pages show a 38.4% decline rate vs. 37.5% for non-flagged pages (n=41,169 vs. n=135,569) — less than a 1-point gap. Verdict: FALSE — the flag doesn't meaningfully separate declining from non-declining pages in this month's data.

What this means: low_ctr_visible_page may still be a valid CTR-optimization candidate flag (it correctly finds pages with a real click-through gap relative to their position, per Signal 2), but it is not a reliable decline-prediction signal. These are two different questions — "is this page under-capturing clicks for its position" versus "is this page trending downward" — and the flag answers the first, not the second. Using it as a proxy for "this page needs urgent review" conflates two separate problems.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

None of the three signals I tested (staleness, CTR-position, search volume) show a strong, reliable link to whether a page's traffic is trending up or down within a single month — the clearest finding was that FlyRank's low_ctr_visible_page flag, while correctly identifying pages with a real click-through gap for their position, barely separates declining from non-declining pages (38.4% vs. 37.5%). This suggests that "low CTR for your position" and "this page is declining" are two different problems that shouldn't be treated as the same signal.

For the content team, this means single-signal flags (age, CTR, volume alone) aren't strong enough on their own to prioritize a review queue — they're each true statements about a page's current state, but weak predictors of where it's headed. A more reliable approach likely needs to combine multiple signals together (which is exactly what Week 5's model is for) rather than trusting any one rule in isolation. In the meantime, low_ctr_visible_page is still worth keeping as a CTR-optimization flag — just not as a decline-risk flag.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.